In [1]:
import pandas as pd

df = pd.read_csv(r'C:\Users\juanb\Desktop\IronHack\Proyecto final\ChurnGuard\data\archive (2)\WA_Fn-UseC_-Telco-Customer-Churn.csv', low_memory=False)


In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

In [4]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_encoded = pd.get_dummies(X, drop_first=True)
Y_encoded = y.map({'No': 0, 'Yes': 1})
print(Y_encoded.head())

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64


In [5]:
bool_cols = X_encoded.select_dtypes(include='bool').columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)
print(X_encoded.dtypes)

SeniorCitizen                              int64
tenure                                     int64
MonthlyCharges                           float64
TotalCharges                             float64
customerID_0003-MKNFE                      int64
                                          ...   
Contract_Two year                          int64
PaperlessBilling_Yes                       int64
PaymentMethod_Credit card (automatic)      int64
PaymentMethod_Electronic check             int64
PaymentMethod_Mailed check                 int64
Length: 7072, dtype: object


In [6]:
from sklearn.model_selection import train_test_split

stratify = Y_encoded
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, Y_encoded, test_size=0.3, random_state=42, stratify=stratify
)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(4930, 7072) (2113, 7072) (4930,) (2113,)


In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Pesos por clase para compensar el desbalance (mismo efecto que class_weight='balanced')
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# Definicion de modelos
lr  = LogisticRegression(class_weight='balanced', random_state=42)
rf  = RandomForestClassifier(class_weight='balanced', random_state=42)
ada = AdaBoostClassifier(random_state=42)
gb  = GradientBoostingClassifier(random_state=42)

# Entrenamiento
lr.fit(X_train_scaled, y_train)
rf.fit(X_train_scaled, y_train)
ada.fit(X_train_scaled, y_train, sample_weight=sample_weights)
gb.fit(X_train_scaled, y_train, sample_weight=sample_weights)

# Predicciones
y_pred_lr  = lr.predict(X_test_scaled)
y_pred_rf  = rf.predict(X_test_scaled)
y_pred_ada = ada.predict(X_test_scaled)
y_pred_gb  = gb.predict(X_test_scaled)

# Resultados
print('Logistic Regression Classification Report:')
print(classification_report(y_test, y_pred_lr))

print('Random Forest Classification Report:')
print(classification_report(y_test, y_pred_rf))

print('AdaBoost Classification Report:')
print(classification_report(y_test, y_pred_ada))

print('Gradient Boosting Classification Report:')
print(classification_report(y_test, y_pred_gb))

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.92      0.87      1552
           1       0.66      0.42      0.51       561

    accuracy                           0.79      2113
   macro avg       0.74      0.67      0.69      2113
weighted avg       0.77      0.79      0.77      2113

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.90      0.87      1552
           1       0.65      0.50      0.57       561

    accuracy                           0.80      2113
   macro avg       0.74      0.70      0.72      2113
weighted avg       0.79      0.80      0.79      2113

AdaBoost Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.73      0.81      1552
           1       0.51      0.77      0.61       561

    accuracy                           0.74      2113
   macro avg    

In [8]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import pandas as pd

modelos = {
    'Logistic Regression': y_pred_lr,
    'Random Forest':       y_pred_rf,
    'AdaBoost':            y_pred_ada,
    'Gradient Boosting':   y_pred_gb,
}

resultados = []
for nombre, y_pred in modelos.items():
    resultados.append({
        'Modelo':     nombre,
        'Precision':  round(precision_score(y_test, y_pred), 3),
        'Recall':     round(recall_score(y_test, y_pred), 3),
        'F1':         round(f1_score(y_test, y_pred), 3),
        'Accuracy':   round(accuracy_score(y_test, y_pred), 3),
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print(df_resultados.to_string())

                     Precision  Recall     F1  Accuracy
Modelo                                                 
Logistic Regression      0.661   0.421  0.514     0.789
Random Forest            0.653   0.499  0.566     0.796
AdaBoost                 0.509   0.766  0.612     0.742
Gradient Boosting        0.519   0.818  0.635     0.751


## Comparativa de modelos y conclusiones

### Criterio de evaluacion

En este problema el coste de un falso negativo (cliente que se va y no detectamos) es de 200 euros, frente a los 5 euros de un falso positivo (llamada innecesaria). La metrica clave es por tanto el recall sobre la clase 1 (churn), que mide directamente cuantos churners reales estamos detectando.

### Logistic Regression

Es el modelo con mayor recall, lo que lo convierte en el mas adecuado para este contexto de negocio. Sacrifica precision (genera mas falsos positivos) pero minimiza los falsos negativos, que son el error mas caro. Es un modelo interpretable y facil de justificar ante negocio.

### Random Forest

Mejora la precision y la accuracy respecto a regresion logistica, pero su recall es notablemente inferior. En terminos economicos, deja escapar a mas clientes que se van sin detectarlos, lo que lo penaliza en este caso de uso concreto.

### AdaBoost

Modelo de boosting secuencial que combina multiples clasificadores debiles. Suele encontrar un equilibrio entre precision y recall, aunque sin ajuste de pesos de clase puede tender a favorecer la clase mayoritaria (no churn). Hay que revisar su recall en los resultados obtenidos para valorar si compensa.

### Gradient Boosting

Generalmente el modelo con mejor F1 de los cuatro, ya que optimiza el error de forma iterativa y captura relaciones no lineales complejas. Sin embargo, si su recall sigue por debajo de regresion logistica, el argumento economico sigue favoreciendo a esta ultima.

### Conclusion

Con un coste de falso negativo 40 veces superior al de falso positivo, el modelo ganador es aquel con mayor recall, independientemente de su accuracy o F1. Logistic Regression con class_weight='balanced' es el punto de partida mas solido. Si se quiere mejorar sin perder recall, el siguiente paso es ajustar el threshold de decision en los modelos de boosting en lugar de usar el 0.5 por defecto.